# Prototype: Erste Exploration der Rohdaten

Dieses Notebook exploriert die NYC Parking Violations Rohdaten auf einem 1%-Sample.

## Ziel

- Rohdaten aus HDFS laden
- 1%-Sample ziehen für schnelle Exploration ohne lange Wartezeiten
- Spaltenstruktur verstehen
- Null-Werte prüfen
- Datums- und Zeitformate prüfen
- mögliche Duplikate erkennen
- Erkenntnisse für Pre-processing ableiten

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, sum as spark_sum

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_RawPrototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 10:47:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Rohdaten laden
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

dfs = []
for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(path, header=True, inferSchema=False) \
        .withColumn("Fiscal Year", lit(fiscal_year))
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

print(f"Gesamtzahl Rohdaten: {df_raw.count():,} Zeilen")
df_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

Gesamtzahl Rohdaten: 54,223,582 Zeilen


26/05/30 10:47:39 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 6:======================================================>  (73 + 3) / 76]

+-----------+--------+
|Fiscal Year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16101101|
|       2025|16559243|
+-----------+--------+



In [3]:
# 1%-Sample ziehen
sample_raw = df_raw.sample(fraction=0.01, seed=42)

sample_count = sample_raw.count()
print(f"Sample-Grösse: {sample_count:,} Zeilen (1% der Rohdaten)")

[Stage 9:=======================================================> (74 + 2) / 76]

Sample-Grösse: 542,412 Zeilen (1% der Rohdaten)


In [4]:
# Schema prüfen
sample_raw.printSchema()

root
 |-- Summons Number: string (nullable = true)
 |-- Plate ID: string (nullable = true)
 |-- Registration State: string (nullable = true)
 |-- Plate Type: string (nullable = true)
 |-- Issue Date: string (nullable = true)
 |-- Violation Code: string (nullable = true)
 |-- Vehicle Body Type: string (nullable = true)
 |-- Vehicle Make: string (nullable = true)
 |-- Issuing Agency: string (nullable = true)
 |-- Street Code1: string (nullable = true)
 |-- Street Code2: string (nullable = true)
 |-- Street Code3: string (nullable = true)
 |-- Vehicle Expiration Date: string (nullable = true)
 |-- Violation Location: string (nullable = true)
 |-- Violation Precinct: string (nullable = true)
 |-- Issuer Precinct: string (nullable = true)
 |-- Issuer Code: string (nullable = true)
 |-- Issuer Command: string (nullable = true)
 |-- Issuer Squad: string (nullable = true)
 |-- Violation Time: string (nullable = true)
 |-- Time First Observed: string (nullable = true)
 |-- Violation County: str

In [5]:
# Null-Werte prüfen
important_columns = [
    "Summons Number",
    "Plate ID",
    "Issue Date",
    "Violation Time",
    "Violation Code",
    "Violation Description",
    "Vehicle Make",
    "Violation County"
]

sample_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in important_columns
]).show(truncate=False)

[Stage 12:=======================================================>(75 + 1) / 76]

+--------------+--------+----------+--------------+--------------+---------------------+------------+----------------+
|Summons Number|Plate ID|Issue Date|Violation Time|Violation Code|Violation Description|Vehicle Make|Violation County|
+--------------+--------+----------+--------------+--------------+---------------------+------------+----------------+
|0             |0       |28        |14            |0             |10626                |545         |7678            |
+--------------+--------+----------+--------------+--------------+---------------------+------------+----------------+



In [6]:
# violation_description: wie viele sind leer oder NULL?
total = sample_raw.count()
empty_desc = sample_raw.filter(
    col("Violation Description").isNull() | (col("Violation Description") == "")
).count()

print(f"Violation Description leer/NULL: {empty_desc:,} von {total:,} ({empty_desc/total*100:.1f}%)")

# Was sind die vorhandenen Werte?
sample_raw.groupBy("Violation Description").count() \
    .orderBy("count", ascending=False).show(10, truncate=False)

Violation Description leer/NULL: 10,626 von 542,412 (2.0%)


[Stage 21:=======================================================>(75 + 1) / 76]

+------------------------------+------+
|Violation Description         |count |
+------------------------------+------+
|PHTO SCHOOL ZN SPEED VIOLATION|185094|
|21-No Parking (street clean)  |46071 |
|38-Failure to Dsplay Meter Rec|38176 |
|14-No Standing                |25668 |
|BUS LANE VIOLATION            |22978 |
|FAILURE TO STOP AT RED LIGHT  |22740 |
|40-Fire Hydrant               |19311 |
|No Parking Street Cleaning    |16726 |
|71A-Insp Sticker Expired (NYS)|16156 |
|20A-No Parking (Non-COM)      |14546 |
+------------------------------+------+
only showing top 10 rows



In [7]:
# Violation Time Format verstehen
sample_raw.select("Violation Time").distinct().show(20, truncate=False)

# Wie viele haben kein gültiges Format?
from pyspark.sql.functions import regexp_extract
sample_raw.filter(
    ~col("Violation Time").rlike(r"^\d{3,4}[AP]$") & col("Violation Time").isNotNull()
).select("Violation Time").distinct().show(20)

+--------------+
|Violation Time|
+--------------+
|0756A         |
|0840P         |
|0449P         |
|1027A         |
|1024A         |
|0713A         |
|0126A         |
|0448A         |
|1010P         |
|0831A         |
|1042A         |
|0135P         |
|0911P         |
|0435P         |
|0519P         |
|0113A         |
|0525P         |
|0840A         |
|0648P         |
|0128P         |
+--------------+
only showing top 20 rows



[Stage 27:======================================================> (74 + 2) / 76]

+--------------+
|Violation Time|
+--------------+
|          0309|
+--------------+



In [8]:
# Issue Date Format prüfen
from pyspark.sql.functions import to_date, year

sample_raw = sample_raw.withColumn(
    "issue_date_parsed",
    to_date(col("Issue Date"), "MM/dd/yyyy")
).withColumn(
    "issue_year",
    year(col("issue_date_parsed"))
)

# Jahresverteilung
sample_raw.groupBy("issue_year").count().orderBy("issue_year").show(30)

[Stage 30:=======================================================>(75 + 1) / 76]

+----------+------+
|issue_year| count|
+----------+------+
|      NULL|    28|
|      2000|     2|
|      2012|     1|
|      2020|     3|
|      2021|     6|
|      2022| 92101|
|      2023|208931|
|      2024|163761|
|      2025| 77572|
|      2026|     3|
|      2027|     1|
|      2028|     1|
|      2029|     1|
|      2052|     1|
+----------+------+



In [9]:
# Duplikate auf summons_number prüfen
from pyspark.sql.functions import upper, trim

total = sample_raw.count()
distinct = sample_raw.select("Summons Number").distinct().count()

print(f"Gesamtzeilen:    {total:,}")
print(f"Distinct summons_number: {distinct:,}")
print(f"Duplikate:       {total - distinct:,}")

[Stage 36:=======================================================>(75 + 1) / 76]

Gesamtzeilen:    542,412
Distinct summons_number: 542,009
Duplikate:       403


## Erkenntnisse aus der Rohdaten-Exploration

Die Exploration des 1%-Samples der Rohdaten (542'412 Zeilen) zeigt folgende Auffälligkeiten:

1. **`Violation Description`:** 10'626 Einträge im Sample haben keine Beschreibung.

2. **`Violation Time` im ungewöhnlichen 12-Stunden-Format:** Das Format ist z.B. `0834A` (08:34 AM) oder `0215P` (14:15 PM). Vereinzelt gibt es ungültige Werte ohne A/P-Suffix (z.B. `0309`). Diese müssen beim Parsen speziell behandelt werden.

3. **Datums-Tippfehler:** Der Datensatz enthält Jahreszahlen weit ausserhalb des erwarteten Bereichs (z.B. 2000, 2012, 2027, 2052). Diese Einträge sind vermutlich Tippfehler und sollten gefiltert werden.

4. **Duplikate:** Im Sample wurden bereits 403 Duplikate auf `summons_number` gefunden. 

Diese Erkenntnisse fliessen direkt ins Pre-processing (`src/1_Pre_Processing/5.0_clean_parking_violations.ipynb`) ein.

In [10]:
spark.stop()